# Cochleogram Preprocessing — Exploration Notebook

This notebook walks through the complete preprocessing pipeline step-by-step:
1. Load a raw ICBHI audio file and parse its annotation.
2. Slice a single respiratory cycle.
3. Visualize the raw waveform.
4. Compare the **mel-spectrogram** baseline vs. the **cochleagram** (pycochleagram).
5. Show the final resized tensor that feeds into the ViT.

**Reference paper:** *Classification of Adventitious Sounds Combining Cochleogram and Vision Transformers*  
**Dataset:** ICBHI 2017 Challenge — https://bhichallenge.med.auth.gr/

In [ ]:
import sys
sys.path.insert(0, '../src')  # make cochleogram_vit importable

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchaudio
import librosa
import librosa.display

from cochleogram_vit.data.dataset import build_icbhi_dataframe, CLASS_NAMES
from cochleogram_vit.preprocessing.cochleogram import CochleogramTransform
from cochleogram_vit.utils.config import load_config

cfg = load_config('../configs/default.yaml')
RAW_DIR = Path('../data/raw')
SR = cfg['data']['sample_rate']
CLIP_DURATION = cfg['data']['clip_duration']
print('Config loaded. Target SR:', SR)

## 1. Parse the dataset

In [ ]:
df = build_icbhi_dataframe(RAW_DIR)
print(f'Total respiratory cycles: {len(df)}')
print('\nLabel distribution:')
print(df['label_name'].value_counts())
df.head(10)

## 2. Load and visualize a single cycle

In [ ]:
# Pick one example of each class
examples = {name: df[df['label_name'] == name].iloc[0] for name in CLASS_NAMES if name in df['label_name'].values}

row = examples.get('crackle', df.iloc[0])  # fallback to first row
print(f'Selected: {row["wav_path"]}  [{row["start"]:.2f}s – {row["end"]:.2f}s]  label={row["label_name"]}')

waveform, orig_sr = torchaudio.load(row['wav_path'])
if waveform.shape[0] > 1:
    waveform = waveform.mean(0, keepdim=True)
if orig_sr != SR:
    waveform = torchaudio.transforms.Resample(orig_sr, SR)(waveform)

start_s = int(row['start'] * SR)
end_s   = int(row['end'] * SR)
clip    = waveform[:, start_s:end_s]

clip_len = int(CLIP_DURATION * SR)
if clip.shape[-1] < clip_len:
    clip = torch.nn.functional.pad(clip, (0, clip_len - clip.shape[-1]))
else:
    clip = clip[..., :clip_len]

audio_np = clip.squeeze().numpy()
time_axis = np.linspace(0, CLIP_DURATION, len(audio_np))

plt.figure(figsize=(12, 2))
plt.plot(time_axis, audio_np, linewidth=0.5)
plt.title(f'Waveform — {row["label_name"]} ({Path(row["wav_path"]).stem})')
plt.xlabel('Time (s)'); plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()

## 3. Mel-Spectrogram baseline

In [ ]:
mel = librosa.feature.melspectrogram(y=audio_np, sr=SR, n_mels=128, fmin=50, fmax=8000)
log_mel = librosa.power_to_db(mel, ref=np.max)

fig, ax = plt.subplots(figsize=(10, 4))
img = librosa.display.specshow(log_mel, sr=SR, x_axis='time', y_axis='mel',
                                fmin=50, fmax=8000, ax=ax, cmap='magma')
fig.colorbar(img, ax=ax, format='%+2.0f dB')
ax.set_title(f'Mel-Spectrogram — {row["label_name"]}')
plt.tight_layout()
plt.show()

## 4. Cochleagram

If `pycochleagram` is installed the cell below uses the full biologically-inspired gammatone filter bank.  
Otherwise it falls back to the librosa mel-spectrogram (logged as a warning).

In [ ]:
cg_cfg = cfg['cochleogram']
transform = CochleogramTransform(
    sr=SR,
    n_filters=cg_cfg['n_filters'],
    low_lim=cg_cfg['low_lim'],
    high_lim=cg_cfg['high_lim'],
    sample_factor=cg_cfg['sample_factor'],
    downsample=cg_cfg.get('downsample'),
    output_size=cfg['model']['image_size'],
)

cg_tensor = transform(clip)   # (1, 128, 128)
cg_np = cg_tensor.squeeze().numpy()

plt.figure(figsize=(6, 5))
plt.imshow(cg_np, origin='lower', aspect='auto', cmap='magma', vmin=0, vmax=1)
plt.colorbar(label='Normalized amplitude')
plt.title(f'Cochleagram (128×128) — {row["label_name"]}')
plt.xlabel('Time frames'); plt.ylabel('ERB frequency channels')
plt.tight_layout()
plt.show()

print('Output tensor shape:', cg_tensor.shape, '  dtype:', cg_tensor.dtype)

## 5. Side-by-side comparison across classes

In [ ]:
fig, axes = plt.subplots(1, len(examples), figsize=(4 * len(examples), 4))
if len(examples) == 1:
    axes = [axes]

for ax, (label_name, ex_row) in zip(axes, examples.items()):
    wf, sr_orig = torchaudio.load(ex_row['wav_path'])
    if wf.shape[0] > 1:
        wf = wf.mean(0, keepdim=True)
    if sr_orig != SR:
        wf = torchaudio.transforms.Resample(sr_orig, SR)(wf)
    s = int(ex_row['start'] * SR)
    e = int(ex_row['end'] * SR)
    c = wf[:, s:e]
    if c.shape[-1] < clip_len:
        c = torch.nn.functional.pad(c, (0, clip_len - c.shape[-1]))
    else:
        c = c[..., :clip_len]
    cg = transform(c).squeeze().numpy()
    ax.imshow(cg, origin='lower', aspect='auto', cmap='magma', vmin=0, vmax=1)
    ax.set_title(label_name)
    ax.set_xlabel('Time'); ax.set_ylabel('ERB channel')

fig.suptitle('Cochleograms by class', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Batch shape check — ViT input compatibility

In [ ]:
import sys
sys.path.insert(0, '../src')
from cochleogram_vit.models.vit import CochleogramViT

model = CochleogramViT.from_config(cfg)
model.eval()

# Simulate a batch of 4 cochleograms
dummy_batch = torch.randn(4, 1, cfg['model']['image_size'], cfg['model']['image_size'])
with torch.no_grad():
    logits = model(dummy_batch)

print('Input shape :', dummy_batch.shape)
print('Output shape:', logits.shape)  # should be (4, 4)
print('Logits      :', logits)